# Curve fitting to HbA1c with H2O AutoML

Record of the curve-fitting modelling workflow, kept for reference. **This
notebook is illustrative and is not executable in this repository**: it was run
on Google Colab against fitted peak parameters held on Drive, and the fits
themselves are not distributed here.

Peak parameters come from `run_curve_fit.py`, which writes one CSV per sample
named `<index>_fit_<HbA1c>_<age>.csv`, containing wavenumber, FWHM, amplitude,
eta, height, area and area_absolute for every detected peak. Each parameter was
evaluated as a feature set; peak height performed best and is the configuration
reported in the manuscript.

The train/test partition is not re-derived here. It is read from the split
indices written by `preprocessing.ipynb`, so curve fitting, PLSR and the CNN are
scored on exactly the same held-out samples. Those indices address the filtered
cohort of 673, so they are mapped back to original dataset rows before the
corresponding fits are selected.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, r2_score

import h2o
from h2o.automl import H2OAutoML

In [ ]:
PARAMETERS = ('wavenumber', 'FWHM', 'amplitude', 'eta', 'height', 'area', 'area_absolute')
FIT_PATTERN = re.compile(r'^(\d+)_fit_([\d.]+)_([\d.]+)$')


class FitReader:
    """
    Assemble per-sample curve-fit parameters into one table per parameter.

    Each table holds one row per sample and one column per peak, indexed by the
    sample number parsed from the filename. Indexing by sample number rather
    than by read order is what allows an externally defined split to be applied
    exactly.
    """

    def __init__(self, dir_path, peaks_path):
        self.dir_path = Path(dir_path)
        self.peaks = pd.read_csv(peaks_path).to_numpy().reshape(-1).astype(int)
        self.tables = self._load()

    def _load(self):
        records = {name: {} for name in PARAMETERS}
        meta = {}

        for path in self.dir_path.glob('*.csv'):
            match = FIT_PATTERN.match(path.stem)
            if match is None:
                continue

            sample = int(match.group(1))
            meta[sample] = (float(match.group(2)), float(match.group(3)))

            fit = pd.read_csv(path)
            for name in PARAMETERS:
                records[name][sample] = fit[name].to_numpy()

        samples = sorted(meta)

        return {
            name: pd.DataFrame(
                [values[s] for s in samples], columns=self.peaks, index=samples
            ).assign(
                HbA1c=[meta[s][0] for s in samples],
                Age=[meta[s][1] for s in samples],
            )
            for name, values in records.items()
        }

    def __getitem__(self, parameter):
        return self.tables[parameter]

    def split_by_samples(self, train_samples, test_samples,
                         parameter='height', target='HbA1c'):
        """
        Select training and test rows by original sample number.

        Raises if any requested sample has no fit, rather than silently
        scoring on a smaller cohort than the other models.
        """
        table = self.tables[parameter]
        requested = set(train_samples) | set(test_samples)
        missing = sorted(requested - set(table.index))
        if missing:
            raise KeyError(f'no fit for {len(missing)} samples, e.g. {missing[:5]}')

        train, test = table.loc[train_samples], table.loc[test_samples]
        features = [c for c in table.columns if c not in ('HbA1c', 'Age')]
        return train[features], test[features], train[target], test[target]

In [ ]:
FIT_DIR = 'data/fit_681_56/'
PEAKS_PATH = 'data/peaks_56.csv'
OUT_PATH = 'data/curvefit_hba1c'

DATASET_PATH = '../data/dataset_681.csv'
SPLIT_DIR = Path('../data/processed')

TARGET = 'HbA1c'
HBA1C_MAX = 14.0
EXCLUDE_INDICES = [287, 636]
FEATURE_SET = 'height'

In [ ]:
hba1c = pd.read_csv(DATASET_PATH)[TARGET].to_numpy()
retained = np.array([
    i for i in range(len(hba1c))
    if i not in EXCLUDE_INDICES and hba1c[i] <= HBA1C_MAX
])

train_samples = retained[np.concatenate([
    np.load(SPLIT_DIR / 'train_idx.npy'),
    np.load(SPLIT_DIR / 'val_idx.npy')
])]
test_samples = retained[np.load(SPLIT_DIR / 'test_idx.npy')]

print(f'{len(retained)} retained  ->  {len(train_samples)} train, {len(test_samples)} test')

reader = FitReader(FIT_DIR, PEAKS_PATH)
X_train, X_test, y_train, y_test = reader.split_by_samples(
    train_samples, test_samples, parameter=FEATURE_SET, target=TARGET
)

In [ ]:
h2o.init()

train = h2o.H2OFrame(pd.concat([
    X_train.reset_index(drop=True),
    pd.DataFrame(y_train, columns=[TARGET]).reset_index(drop=True)
], axis=1))

test = h2o.H2OFrame(pd.concat([
    X_test.reset_index(drop=True),
    pd.DataFrame(y_test, columns=[TARGET]).reset_index(drop=True)
], axis=1))

predictors = [c for c in train.columns if c != TARGET]

In [ ]:
aml = H2OAutoML(
    max_runtime_secs=1800,
    max_models=50,
    nfolds=5,
    seed=1234,
    exclude_algos=['StackedEnsemble', 'DeepLearning']
)
aml.train(x=predictors, y=TARGET, training_frame=train)

In [ ]:
aml.get_leaderboard().head(20)

In [ ]:
aml.leader.model_performance(test)

In [ ]:
predicted = aml.leader.predict(test)
result = pd.concat([
    test[TARGET].as_data_frame().reset_index(drop=True),
    predicted.as_data_frame().reset_index(drop=True)
], axis=1)

mse = mean_squared_error(result[TARGET], result['predict'])
print(f"R2   {r2_score(result[TARGET], result['predict']):.3f}")
print(f"R    {pearsonr(result[TARGET], result['predict'])[0]:.3f}")
print(f"MAE  {np.abs(result[TARGET] - result['predict']).mean():.3f}")
print(f"MSE  {mse:.3f}")
print(f"RMSE {np.sqrt(mse):.3f}")

result.to_csv(os.path.join(OUT_PATH, f'predict_curvefit_{FEATURE_SET}.csv'), index=None)

In [ ]:
models = aml.leaderboard.head(20)['model_id'].as_data_frame().values.flatten()

variable_importances = []
for model_id in models:
    model = h2o.get_model(model_id)
    if hasattr(model, 'varimp'):
        varimp = model.varimp(use_pandas=True)
        varimp['model_id'] = model_id
        variable_importances.append(varimp)

all_varimp = pd.concat(variable_importances)
top_features = all_varimp.groupby('variable')['scaled_importance'].mean().nlargest(10).index
top_varimp = all_varimp[all_varimp['variable'].isin(top_features)]

top_varimp.to_csv(os.path.join(OUT_PATH, f'top_varimp_{FEATURE_SET}.csv'), index=None)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=top_varimp, x='variable', y='scaled_importance', ax=ax)
ax.set_xlabel('Peak position (cm⁻¹)')
ax.set_ylabel('Scaled importance')
ax.tick_params(axis='x', rotation=45)

In [ ]:
res = aml.leader.shap_summary_plot(train)
res.figure().savefig(os.path.join(OUT_PATH, f'SHAP_{FEATURE_SET}.png'), dpi=300)

h2o.save_model(model=aml.leader, path=OUT_PATH, force=True)